
# GPTQ full-quant notebook — GSM8K + ARC-Challenge only

This notebook does exactly three experiment steps for **Qwen2-1.5B full GPTQ quantization**:

1. quantize and save the model artifact
2. evaluate on **GSM8K**
3. evaluate on **ARC-Challenge**

It also keeps a compact experiment summary with the table-style metrics you wanted for the **full quant** row:

- Variant
- Compression
- Model Size (GB)
- Peak VRAM (GB)
- Bits/Param
- GSM8K Accuracy
- ARC Accuracy

The dataset settings are kept aligned with the earlier notebook:

- **GSM8K:** 8-shot, 300 samples, max_new_tokens=256
- **ARC-Challenge:** 500 samples, max_new_tokens=5


In [ ]:
# Colab / notebook setup
!pip install -U pip setuptools wheel
!pip install -U gptqmodel
!pip install -q torch transformers accelerate datasets pandas pyarrow safetensors tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.0 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.4/668.4 kB 33.5 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 228.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 148.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 222.5 MB/s  0:00:00
   ━━━━━━━━━━━━━

In [ ]:

# Restart once after installation in a fresh runtime, then skip this cell on reruns.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


{'status': 'ok', 'restart': True}

In [ ]:
def get_calibration_data(cfg, tokenizer=None):
    cal_cfg = cfg['calibration']
    cache_path = CALIB_DIR / f"calibration_{cal_cfg['num_samples']}_{cal_cfg['max_length']}.pt"
    if cache_path.exists():
        logger.info('Using cached calibration samples: %s', cache_path)
        return torch.load(cache_path, weights_only=False)

    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'], trust_remote_code=True, use_fast=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

    logger.info('Loading calibration dataset: %s / %s / %s', cal_cfg['dataset'], cal_cfg['dataset_name'], cal_cfg['split'])
    ds = load_dataset(cal_cfg['dataset'], cal_cfg['dataset_name'], split=cal_cfg['split'])
    all_text = '\n\n'.join(t for t in ds['text'] if t and t.strip())
    encoded = tokenizer(all_text, return_tensors='pt')
    total_tokens = encoded.input_ids.shape[1]
    max_len = cal_cfg['max_length']

    if total_tokens <= max_len:
        raise ValueError(f'Calibration corpus too short: total_tokens={total_tokens}, max_length={max_len}')

    rng = torch.Generator().manual_seed(cal_cfg['seed'])
    starts = torch.randint(0, total_tokens - max_len, (cal_cfg['num_samples'],), generator=rng)
    samples = []
    for start in tqdm(starts.tolist(), desc='Preparing calibration windows', unit='sample'):
        end = start + max_len
        input_ids = encoded.input_ids[:, start:end]
        samples.append({
            'input_ids': input_ids,
            'attention_mask': torch.ones_like(input_ids),
        })

    torch.save(samples, cache_path)
    logger.info('Saved calibration cache: %s', cache_path)
    return samples


def quantize_full_gptq(cfg):
    tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'], trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantized_already = (ARTIFACT_DIR / 'config.json').exists() and find_gptq_model_basename(ARTIFACT_DIR) is not None
    if quantized_already:
        logger.info('Quantized artifact already exists at %s', ARTIFACT_DIR)
        zip_path = BUNDLE_DIR / 'full_quant_quantized_model.zip'
        if not zip_path.exists():
            zip_directory(ARTIFACT_DIR, zip_path)
        return {
            'artifact_dir': str(ARTIFACT_DIR),
            'zip_path': str(zip_path),
            'gptq_model_file': find_gptq_model_basename(ARTIFACT_DIR),
            'model_size_gb': round(model_size_gb(ARTIFACT_DIR), 4),
            'bits_per_param': round(bits_per_param(ARTIFACT_DIR), 4),
        }

    samples = get_calibration_data(cfg, tokenizer)
    cal_dataset = [
        {
            'input_ids': s['input_ids'].squeeze(0),
            'attention_mask': s['attention_mask'].squeeze(0),
        }
        for s in samples
    ]

    quantize_config = QuantizeConfig(
        bits=cfg['quant']['bits'],
        group_size=cfg['quant']['group_size'],
        desc_act=cfg['quant'].get('desc_act', False),
    )

    logger.info('Starting GPTQ full quantization for %s', cfg['base_model'])
    t0 = time.time()
    model = GPTQModel.load(cfg['base_model'], quantize_config, trust_remote_code=True)
    model.quantize(cal_dataset)
    model.save(str(ARTIFACT_DIR))
    tokenizer.save_pretrained(str(ARTIFACT_DIR))
    elapsed = time.time() - t0
    logger.info('Quantization finished in %s', format_seconds(elapsed))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_file = find_gptq_model_basename(ARTIFACT_DIR)
    if model_file is None:
        raise FileNotFoundError(f'No GPTQ safetensors found in {ARTIFACT_DIR}')

    manifest = {
        'variant': VARIANT,
        'base_model': cfg['base_model'],
        'quant': cfg['quant'],
        'artifact_dir': str(ARTIFACT_DIR),
        'gptq_model_file': model_file,
        'quantization_elapsed_sec': elapsed,
        'model_size_gb': round(model_size_gb(ARTIFACT_DIR), 4),
        'bits_per_param': round(bits_per_param(ARTIFACT_DIR), 4),
        'created_at_unix': time.time(),
    }
    save_json(manifest, ARTIFACT_DIR / 'artifact_manifest.json')
    save_json(manifest, RESULTS_DIR / 'quantization_manifest.json')

    zip_path = zip_directory(ARTIFACT_DIR, BUNDLE_DIR / 'full_quant_quantized_model.zip')

    state = get_results_state()
    state['quantization'] = manifest | {'zip_path': str(zip_path)}
    save_results_state(state)
    summary = update_summary_table(state)

    print('Quantized artifact ready.')
    print(json.dumps(manifest, indent=2))
    display(pd.DataFrame([summary]))
    return manifest | {'zip_path': str(zip_path)}

In [ ]:
import os
import gc
import json
import math
import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from gptqmodel import GPTQModel

# -----------------------------
# Perplexity config
# -----------------------------
PPL_DATASET_NAME = "wikitext"
PPL_DATASET_CONFIG = "wikitext-2-raw-v1"
PPL_SPLIT = "test"
PPL_MAX_LENGTH = 2048
PPL_STRIDE = 1024

print("=" * 80)
print("PERPLEXITY EVALUATION")
print("=" * 80)
print(f"Dataset      : {PPL_DATASET_NAME}/{PPL_DATASET_CONFIG}")
print(f"Split        : {PPL_SPLIT}")
print(f"Max length   : {PPL_MAX_LENGTH}")
print(f"Stride       : {PPL_STRIDE}")
print(f"Artifact dir : {ARTIFACT_DIR}")
print(f"Results dir  : {RESULTS_DIR}")

# -----------------------------
# Reload quantized model
# -----------------------------
print("\n[1/4] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ARTIFACT_DIR, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("[2/4] Loading quantized model...")
torch.cuda.empty_cache()
gc.collect()

model = GPTQModel.load(
    ARTIFACT_DIR,
    device="cuda:0" if torch.cuda.is_available() else "cpu"
)
model.eval()

# -----------------------------
# Load dataset
# -----------------------------
print("[3/4] Loading WikiText-2 test split...")
dataset = load_dataset(PPL_DATASET_NAME, PPL_DATASET_CONFIG, split=PPL_SPLIT)
text = "\n\n".join(dataset["text"])

encodings = tokenizer(text, return_tensors="pt")
input_ids = encodings["input_ids"]

seq_len = input_ids.size(1)
print(f"Tokenized sequence length: {seq_len}")

# -----------------------------
# Sliding-window perplexity
# -----------------------------
print("[4/4] Running sliding-window perplexity evaluation...")

nlls = []
prev_end_loc = 0
num_steps = 0

estimated_total_steps = max(1, math.ceil(max(seq_len - PPL_MAX_LENGTH, 0) / PPL_STRIDE) + 1)
start_time = time.time()

for begin_loc in range(0, seq_len, PPL_STRIDE):
    end_loc = min(begin_loc + PPL_MAX_LENGTH, seq_len)
    trg_len = end_loc - prev_end_loc

    input_ids_slice = input_ids[:, begin_loc:end_loc].to(model.device)
    target_ids = input_ids_slice.clone()
    target_ids[:, :-trg_len] = -100

    with torch.no_grad():
        outputs = model(input_ids=input_ids_slice, labels=target_ids)
        neg_log_likelihood = outputs.loss

    nlls.append(neg_log_likelihood.detach().float() * trg_len)

    prev_end_loc = end_loc
    num_steps += 1

    elapsed = time.time() - start_time
    avg_step = elapsed / num_steps
    eta = avg_step * (estimated_total_steps - num_steps)

    print(
        f"[PPL] step {num_steps}/{estimated_total_steps} | "
        f"window=({begin_loc}, {end_loc}) | "
        f"elapsed={elapsed/60:.2f} min | eta={eta/60:.2f} min"
    )

    if end_loc == seq_len:
        break

ppl = torch.exp(torch.stack(nlls).sum() / end_loc).item()

ppl_metrics = {
    "dataset": f"{PPL_DATASET_NAME}/{PPL_DATASET_CONFIG}",
    "split": PPL_SPLIT,
    "max_length": PPL_MAX_LENGTH,
    "stride": PPL_STRIDE,
    "num_tokens": int(seq_len),
    "num_windows": int(num_steps),
    "perplexity": float(ppl),
}

with open(os.path.join(RESULTS_DIR, "perplexity_metrics.json"), "w") as f:
    json.dump(ppl_metrics, f, indent=2)

with open(os.path.join(RESULTS_DIR, "perplexity_eval_config.json"), "w") as f:
    json.dump({
        "dataset_name": PPL_DATASET_NAME,
        "dataset_config": PPL_DATASET_CONFIG,
        "split": PPL_SPLIT,
        "max_length": PPL_MAX_LENGTH,
        "stride": PPL_STRIDE
    }, f, indent=2)

print("\nPerplexity evaluation complete.")
print(json.dumps(ppl_metrics, indent=2))

PERPLEXITY EVALUATION
Dataset      : wikitext/wikitext-2-raw-v1
Split        : test
Max length   : 2048
Stride       : 1024
Artifact dir : /content/artifacts/gptq_w4_g128/full_quant
Results dir  : /content/results/gptq_w4_g128_full_quant/full_quant

[1/4] Loading tokenizer...
[2/4] Loading quantized model...
from_quantized: adapter: None


INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   


INFO  QuantizeConfig: offload_to_disk_path auto set to `./gptqmodel_offload/bvtyioxl-pcnusonl/`


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO  Kernel: Auto-selection: adding candidate `TritonV2QuantLinear`           


...

INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}



INFO  Kernel: loaded -> `[TritonV2QuantLinear]`                                


[3/4] Loading WikiText-2 test split...


2026-03-09 23:02:49,644 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-09 23:02:49,874 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-09 23:02:49,880 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Salesforce/wikitext/b08601e04326c79dfdd32d625aee71d232d685c3/README.md "HTTP/1.1 200 OK"
2026-03-09 23:02:50,119 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 307 Temporary Redirect"
2026-03-09 23:02:50,346 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 404 Not Found"
2026-03-09 23:02:51,037 | INFO    | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/d

Tokenized sequence length: 299078
[4/4] Running sliding-window perplexity evaluation...
[PPL] step 1/292 | window=(0, 2048) | elapsed=0.00 min | eta=0.94 min
[PPL] step 2/292 | window=(1024, 3072) | elapsed=0.00 min | eta=0.68 min
[PPL] step 3/292 | window=(2048, 4096) | elapsed=0.01 min | eta=0.75 min
[PPL] step 4/292 | window=(3072, 5120) | elapsed=0.01 min | eta=0.80 min
[PPL] step 5/292 | window=(4096, 6144) | elapsed=0.01 min | eta=0.82 min
[PPL] step 6/292 | window=(5120, 7168) | elapsed=0.02 min | eta=0.85 min
[PPL] step 7/292 | window=(6144, 8192) | elapsed=0.02 min | eta=0.86 min
[PPL] step 8/292 | window=(7168, 9216) | elapsed=0.02 min | eta=0.87 min
[PPL] step 9/292 | window=(8192, 10240) | elapsed=0.03 min | eta=0.87 min
[PPL] step 10/292 | window=(9216, 11264) | elapsed=0.03 min | eta=0.88 min
[PPL] step 11/292 | window=(10240, 12288) | elapsed=0.03 min | eta=0.88 min
[PPL] step 12/292 | window=(11264, 13312) | elapsed=0.04 min | eta=0.88 min
[PPL] step 13/292 | window=(12

In [ ]:

# Cell 1 — Quantize the model and save the quantized artifact zip
quant_manifest = quantize_full_gptq(cfg)


2026-03-09 22:16:54,575 | INFO    | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 22:16:54,576 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-09 22:16:54,582 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"
2026-03-09 22:16:54,590 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

2026-03-09 22:16:54,833 | INFO    | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 22:16:54,839 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-09 22:16:54,846 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-03-09 22:16:55,094 | INFO    | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2-1.5B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-09 22:16:55,332 | INFO    | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2-1.5B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-09 22:16:55,565 | INFO    | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 22:16:55,571 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/vocab.json "HTTP/1.1 200 OK"
2026-03-09 22:16:55,578 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/vocab.json "HTTP/1.1 200 OK"


...

INFO  | Process finalize   | 2      | 0.000  | 0.000 | 0.001   | 0.0%   | tp-pre-pad                                        |


INFO  +--------------------+--------+--------+-------+---------+--------+---------------------------------------------------+


2026-03-09 22:33:44,810 | INFO    | Quantization finished in 16:25
2026-03-09 22:35:12,814 | INFO    | Created zip: /content/zip_bundles/gptq_w4_g128_full_quant/full_quant/full_quant_quantized_model.zip (910.57 MB)


Quantized artifact ready.
{
  "variant": "full_quant",
  "base_model": "Qwen/Qwen2-1.5B",
  "quant": {
    "bits": 4,
    "group_size": 128,
    "desc_act": false
  },
  "artifact_dir": "/content/artifacts/gptq_w4_g128/full_quant",
  "gptq_model_file": "model.safetensors",
  "quantization_elapsed_sec": 985.466757774353,
  "model_size_gb": 1.0709,
  "bits_per_param": 5.5627,
  "created_at_unix": 1773095624.8650641
}


,variant,compression,model_size_gb,peak_vram_gb,bits_per_param,gsm8k_accuracy,arc_accuracy
0,full_quant,GPTQ W4 g128,1.0709,None,5.5627,None,None


In [ ]:
# Evaluation helpers for GSM8K and ARC-Challenge only

GSM_PROMPT = "Question: {question} Answer: Let's think step by step."


def evaluate_gsm8k_only(model, tokenizer, cfg):
    gsm_cfg = cfg['eval']['gsm8k']
    n_shot = gsm_cfg['num_fewshot']
    num_samples = gsm_cfg['num_samples']
    max_new = gsm_cfg['max_new_tokens']
    log_every = gsm_cfg.get('log_every', 25)

    logger.info('Loading GSM8K: %d-shot, %d samples, max_new_tokens=%d', n_shot, num_samples, max_new)
    train_ds = load_dataset('openai/gsm8k', 'main', split='train').shuffle(seed=42)
    test_ds = load_dataset('openai/gsm8k', 'main', split='test').shuffle(seed=42).select(range(num_samples))
    exemplars = train_ds.select(range(n_shot))

    prefix = ''
    for ex in exemplars:
        prefix += GSM_PROMPT.format(question=ex['question']) + ex['answer'] + '\n\n'

    rows = []
    correct = 0
    start_time = time.time()
    peak_vram = 0.0
    total = len(test_ds)

    for idx, row in enumerate(tqdm(test_ds, desc='GSM8K', unit='q'), start=1):
        prompt = prefix + GSM_PROMPT.format(question=row['question'])

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        generated = generate_text(model, tokenizer, prompt, max_new_tokens=max_new)
        pred = extract_numeric_answer(generated)
        gold = extract_numeric_answer(row['answer'])
        is_correct = pred is not None and gold is not None and abs(pred - gold) < 1e-3
        correct += int(is_correct)

        if torch.cuda.is_available():
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / (1024 ** 3))

        rows.append({
            'index': idx - 1,
            'question': row['question'],
            'prediction_text': generated,
            'predicted_value': pred,
            'gold_value': gold,
            'correct': bool(is_correct),
        })

        if idx % log_every == 0 or idx == total:
            log_progress('GSM8K', idx, total, start_time, correct)

    elapsed = time.time() - start_time
    result = {
        'dataset': 'gsm8k',
        'num_fewshot': n_shot,
        'num_samples': total,
        'max_new_tokens': max_new,
        'accuracy': correct / max(total, 1),
        'correct': correct,
        'elapsed_sec': elapsed,
        'elapsed_hms': format_seconds(elapsed),
        'peak_vram_gb': peak_vram if torch.cuda.is_available() else None,
    }

    pd.DataFrame(rows).to_csv(RESULTS_DIR / 'gsm8k_predictions.csv', index=False)
    save_json(result, RESULTS_DIR / 'gsm8k_metrics.json')

    state = get_results_state()
    state['gsm8k'] = result
    save_results_state(state)
    summary = update_summary_table(state)

    logger.info('GSM8K finished | acc=%.4f | elapsed=%s | peak_vram_gb=%s', result['accuracy'], result['elapsed_hms'], result['peak_vram_gb'])
    display(pd.DataFrame([result]))
    display(pd.DataFrame([summary]))
    return result


def format_arc_prompt(row):
    labels = row['choices']['label']
    texts = row['choices']['text']
    prompt = row['question'] + '\n'
    for lbl, txt in zip(labels, texts):
        prompt += f'{lbl}. {txt}\n'
    prompt += 'Answer:'
    return prompt


def evaluate_arc_only(model, tokenizer, cfg):
    arc_cfg = cfg['eval']['arc_challenge']
    num_samples = arc_cfg['num_samples']
    max_new = arc_cfg['max_new_tokens']
    log_every = arc_cfg.get('log_every', 50)

    logger.info('Loading ARC-Challenge: %d samples, max_new_tokens=%d', num_samples, max_new)
    ds = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test').shuffle(seed=42).select(range(num_samples))

    rows = []
    correct = 0
    start_time = time.time()
    peak_vram = 0.0
    total = len(ds)

    for idx, row in enumerate(tqdm(ds, desc='ARC-Challenge', unit='q'), start=1):
        prompt = format_arc_prompt(row)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        generated = generate_text(model, tokenizer, prompt, max_new_tokens=max_new).strip()
        pred = generated[:1].upper() if generated else None
        gold = row['answerKey'].strip().upper()
        is_correct = pred == gold
        correct += int(is_correct)

        if torch.cuda.is_available():
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / (1024 ** 3))

        rows.append({
            'index': idx - 1,
            'question': row['question'],
            'prediction_text': generated,
            'predicted_label': pred,
            'gold_label': gold,
            'correct': bool(is_correct),
        })

        if idx % log_every == 0 or idx == total:
            log_progress('ARC-Challenge', idx, total, start_time, correct)

    elapsed = time.time() - start_time
    result = {
        'dataset': 'arc_challenge',
        'num_samples': total,
        'max_new_tokens': max_new,
        'accuracy': correct / max(total, 1),
        'correct': correct,
        'elapsed_sec': elapsed,
        'elapsed_hms': format_seconds(elapsed),
        'peak_vram_gb': peak_vram if torch.cuda.is_available() else None,
    }

    pd.DataFrame(rows).to_csv(RESULTS_DIR / 'arc_challenge_predictions.csv', index=False)
    save_json(result, RESULTS_DIR / 'arc_challenge_metrics.json')

    state = get_results_state()
    state['arc_challenge'] = result
    save_results_state(state)
    summary = update_summary_table(state)

    results_zip = zip_directory(RESULTS_DIR, BUNDLE_DIR / 'full_quant_results_only.zip')
    state['results_zip'] = str(results_zip)
    save_results_state(state)

    logger.info('ARC-Challenge finished | acc=%.4f | elapsed=%s | peak_vram_gb=%s', result['accuracy'], result['elapsed_hms'], result['peak_vram_gb'])
    display(pd.DataFrame([result]))
    display(pd.DataFrame([summary]))
    print('Results zip:', results_zip)
    return result

In [ ]:

# Cell 2 — Evaluate on GSM8K only
# Run this after the quantization cell.
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Quantized artifact not found: {ARTIFACT_DIR}')

model, tokenizer = load_quantized_model_and_tokenizer(ARTIFACT_DIR)
try:
    gsm8k_result = evaluate_gsm8k_only(model, tokenizer, cfg)
finally:
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [14]:

# Cell 3 — Evaluate on ARC-Challenge only and finalize the summary/results zip
# Run this after the GSM8K cell.
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Quantized artifact not found: {ARTIFACT_DIR}')

model, tokenizer = load_quantized_model_and_tokenizer(ARTIFACT_DIR)
try:
    arc_result = evaluate_arc_only(model, tokenizer, cfg)
finally:
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


from_quantized: adapter: None


INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   


INFO  QuantizeConfig: offload_to_disk_path auto set to `./gptqmodel_offload/pnnvijib-iqbxrhpb/`


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO  Kernel: Auto-selection: adding candidate `TritonV2QuantLinear`           


...

ARC-Challenge:   0%|          | 0/500 [00:00<?, ?q/s]

2026-03-09 23:09:33,249 | INFO    | ARC-Challenge | 50/500 | acc=0.6200 | elapsed=0:18 | eta=2:49 | 0.38s/item
2026-03-09 23:09:51,940 | INFO    | ARC-Challenge | 100/500 | acc=0.6500 | elapsed=0:37 | eta=2:30 | 0.38s/item
2026-03-09 23:10:10,604 | INFO    | ARC-Challenge | 150/500 | acc=0.6333 | elapsed=0:56 | eta=2:11 | 0.37s/item
2026-03-09 23:10:29,167 | INFO    | ARC-Challenge | 200/500 | acc=0.6200 | elapsed=1:14 | eta=1:52 | 0.37s/item
2026-03-09 23:10:47,753 | INFO    | ARC-Challenge | 250/500 | acc=0.6480 | elapsed=1:33 | eta=1:33 | 0.37s/item
2026-03-09 23:11:06,425 | INFO    | ARC-Challenge | 300/500 | acc=0.6433 | elapsed=1:51 | eta=1:14 | 0.37s/item
2026-03-09 23:11:25,032 | INFO    | ARC-Challenge | 350/500 | acc=0.6371 | elapsed=2:10 | eta=0:55 | 0.37s/item
2026-03-09 23:11:43,613 | INFO    | ARC-Challenge | 400/500 | acc=0.6325 | elapsed=2:29 | eta=0:37 | 0.37s/item
2026-03-09 23:12:02,212 | INFO    | ARC-Challenge | 450/500 | acc=0.6244 | elapsed=2:47 | eta=0:18 | 0.37

,dataset,num_samples,max_new_tokens,accuracy,correct,elapsed_sec,elapsed_hms,peak_vram_gb
0,arc_challenge,500,5,0.618,309,186.393611,3:06,6.756505


,variant,compression,model_size_gb,peak_vram_gb,bits_per_param,gsm8k_accuracy,arc_accuracy
0,full_quant,GPTQ W4 g128,1.0709,6.7565,5.5627,None,0.618


Results zip: /content/zip_bundles/gptq_w4_g128_full_quant/full_quant/full_quant_results_only.zip


In [16]:
import os
import json
import pandas as pd

summary = {
    "variant": "full_quant",
    "compression": "GPTQ",
    "model_size_gb": float(quant_manifest['model_size_gb']),
    "peak_vram_gb": float(arc_result['peak_vram_gb']),
    "bits_per_param": float(quant_manifest['bits_per_param']),
    "perplexity": None,
    "arc_accuracy": None,
}

ppl_path = os.path.join(RESULTS_DIR, "perplexity_metrics.json")
arc_path = os.path.join(RESULTS_DIR, "arc_challenge_metrics.json")

if os.path.exists(ppl_path):
    with open(ppl_path, "r") as f:
        ppl_data = json.load(f)
    summary["perplexity"] = ppl_data.get("perplexity")


if os.path.exists(arc_path):
    with open(arc_path, "r") as f:
        arc_data = json.load(f)
    summary["arc_accuracy"] = arc_data.get("accuracy")

summary_json_path = os.path.join(RESULTS_DIR, "summary.json")
summary_csv_path = os.path.join(RESULTS_DIR, "summary.csv")

with open(summary_json_path, "w") as f:
    json.dump(summary, f, indent=2)

pd.DataFrame([summary]).to_csv(summary_csv_path, index=False)

print(pd.DataFrame([summary]))

      variant compression  model_size_gb  peak_vram_gb  bits_per_param  \
0  full_quant        GPTQ         1.0709      6.756505          5.5627   

   perplexity  arc_accuracy  
0    8.894765         0.618  
